In [1]:
import pandas as pd
import duckdb
import time


In [2]:
df = pd.read_csv("/home/leloc/Document/USTH/Thesis/Machine_translation-Thai-Viet-/final_data.csv")

In [4]:
import time
import duckdb

con = duckdb.connect("translation.db")

# Tạo bảng nếu chưa có, bỏ dấu phẩy cuối cùng
con.execute("""
    CREATE TABLE IF NOT EXISTS translations (
        thai TEXT,
        vietnamese TEXT
    )
""")

# Chỉ lấy 2 cột cần thiết để chèn
df_subset = df[['Thai', 'Viet']]

start = time.time()

con.register("df_view", df_subset)
con.execute("INSERT INTO translations SELECT Thai, Viet FROM df_view")

end = time.time()
print(f"✅ Đã chèn {len(df_subset)} dòng vào DuckDB trong {end - start:.2f} giây")

con.close()


✅ Đã chèn 306632 dòng vào DuckDB trong 2.69 giây


In [7]:
con = duckdb.connect("translation.db")
print(con.execute("SELECT COUNT(*) FROM translations").fetchone()[0])



306632


In [8]:
df = con.execute("SELECT * FROM translations LIMIT 5").fetchdf()
df

,thai,vietnamese
0,ฉันรู้ว่าคุณเคยบอกว่า จะคอยส่งข่าวฉันเรื่อยๆ แ...,Tôi biết bà đã nói sẽ báo cho tôi nhưng tuần n...
1,เคียว ฉันก็มี มีด ฉันก็มี ดูของฉันซะก่อน,Đủ loại đặc thù nhé Có lưỡi hái tử thần Lưỡi h...
2,แดเนียล เลอรอย แม็คแคบบี้ ที่ 3 สามีฉันเอง,Chắc cậu có nghe nói đến anh ấy Anh ấy phát mi...
3,แพร์รี่ มันยุค 90 นะ ใครสนล่ะ,Thập niên 90 rồi ai quan tâm nữa
4,เธ เธฑเธ เธ เน เธฐ เธ เน เธฒเธ เน เธญเธขเธฒเธ ...,Nhìn xem tôi cũng muốn rời khỏi đây Thật đấy


In [9]:
con.execute("ALTER TABLE translations ADD COLUMN IF NOT EXISTS thai_input_ids BLOB")
con.execute("ALTER TABLE translations ADD COLUMN IF NOT EXISTS thai_attention_mask BLOB")
con.execute("ALTER TABLE translations ADD COLUMN IF NOT EXISTS vi_input_ids BLOB")
con.execute("ALTER TABLE translations ADD COLUMN IF NOT EXISTS vi_attention_mask BLOB")

con.close()

In [10]:
con = duckdb.connect("/home/leloc/Document/USTH/Thesis/Machine_translation-Thai-Viet-/translation.db")
result = con.execute("SHOW TABLES").fetchall()
print(result)

[('translations',)]


In [11]:
result = con.execute("DESCRIBE translations").fetchall()
print(result)

[('thai', 'VARCHAR', 'YES', None, None, None), ('vietnamese', 'VARCHAR', 'YES', None, None, None), ('thai_input_ids', 'BLOB', 'YES', None, None, None), ('thai_attention_mask', 'BLOB', 'YES', None, None, None), ('vi_input_ids', 'BLOB', 'YES', None, None, None), ('vi_attention_mask', 'BLOB', 'YES', None, None, None)]
